# RFD Urban Digital Twin Analysis

This notebook builds compact exam pipeline for RFD-based consistency profiling on urban air-quality data.

Scope:
- simplified conceptual Urban Digital Twin;
- Beijing air-quality subset with two stations;
- interpretable approximate rules, not causal claims;
- lightweight discovery over reduced deterministic sample.


## Dataset Configuration

Selected subset:
- stations: `Aotizhongxin`, `Changping`
- period: `2013-09-01` to `2013-12-31`
- cleaned dataset target: around 5k-6k rows
- RFD experiment sample: 1500 rows balanced across stations

Why December included: Sep-Nov cleaned subset produced only 3996 rows after dropping missing values. Adding December yields 5426 rows.


In [1]:
from pathlib import Path
import sys

import pandas as pd

try:
    from IPython.display import display
except ImportError:
    def display(obj):
        print(obj)

current = Path.cwd().resolve()
PROJECT_ROOT = next(path for path in [current, *current.parents] if (path / 'AGENTS.md').exists())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_RAW = PROJECT_ROOT / 'data' / 'raw'
DATA_PROCESSED = PROJECT_ROOT / 'data' / 'processed'
RESULTS_DIR = PROJECT_ROOT / 'results'
FIGURES_DIR = PROJECT_ROOT / 'figures'

for path in [DATA_PROCESSED, RESULTS_DIR, FIGURES_DIR]:
    path.mkdir(parents=True, exist_ok=True)


In [2]:
from src.experiments import (
    prepare_rfd_sample,
    run_candidate_validation,
    run_lightweight_discovery,
    run_station_comparison,
    run_threshold_comparison,
    select_top_rule_violations,
    export_violation_examples,
)
from src.preprocessing import prepare_udt_dataset
from src.profiling import (
    correlation_matrix,
    data_types_summary,
    dataset_shape_summary,
    hour_distribution,
    missing_values_summary,
    numeric_summary,
    station_distribution,
    time_slot_distribution,
    unique_values_summary,
)
from src.visualization import (
    plot_correlation_matrix,
    plot_missing_values,
    plot_pm25_timeseries,
    plot_pollutant_distribution,
)


## Step 1-2: Preprocessing

In [3]:
cleaned_df, pre_drop_missing = prepare_udt_dataset(
    raw_dir=DATA_RAW,
    processed_path=DATA_PROCESSED / 'udt_rfd_dataset.csv',
)
sample_df = prepare_rfd_sample(cleaned_df)
sample_df.to_csv(DATA_PROCESSED / 'udt_rfd_sample.csv', index=False)

print('Cleaned rows:', len(cleaned_df))
print('Cleaned columns:', cleaned_df.shape[1])
print('RFD sample rows:', len(sample_df))
display(pre_drop_missing)


Cleaned rows: 5426
Cleaned columns: 11
RFD sample rows: 1500


,column,missing_count,missing_rate
0,O3,390,0.066598
1,NO2,32,0.005464
2,PM2.5,7,0.001195
3,PM10,3,0.000512
4,DEWP,0,0.000000
5,TEMP,0,0.000000
6,WSPM,0,0.000000
7,datetime,0,0.000000
8,hour,0,0.000000
9,station,0,0.000000


## Step 3: Profiling

In [4]:
shape_df = dataset_shape_summary(cleaned_df)
dtypes_df = data_types_summary(cleaned_df)
missing_df = missing_values_summary(cleaned_df)
unique_df = unique_values_summary(cleaned_df)
numeric_df = numeric_summary(cleaned_df)
station_df = station_distribution(cleaned_df)
hour_df = hour_distribution(cleaned_df)
time_slot_df = time_slot_distribution(cleaned_df)
corr_df = correlation_matrix(
    cleaned_df,
    numeric_columns=['hour', 'PM2.5', 'PM10', 'NO2', 'O3', 'TEMP', 'DEWP', 'WSPM'],
)

shape_df.to_csv(RESULTS_DIR / 'profile_shape_summary.csv', index=False)
dtypes_df.to_csv(RESULTS_DIR / 'profile_dtypes_summary.csv', index=False)
missing_df.to_csv(RESULTS_DIR / 'profile_missing_summary.csv', index=False)
unique_df.to_csv(RESULTS_DIR / 'profile_unique_values_summary.csv', index=False)
numeric_df.to_csv(RESULTS_DIR / 'profile_numeric_summary.csv', index=False)
station_df.to_csv(RESULTS_DIR / 'profile_station_distribution.csv', index=False)
hour_df.to_csv(RESULTS_DIR / 'profile_hour_distribution.csv', index=False)
time_slot_df.to_csv(RESULTS_DIR / 'profile_time_slot_distribution.csv', index=False)
corr_df.to_csv(RESULTS_DIR / 'profile_correlation_matrix.csv')

plot_missing_values(missing_df, FIGURES_DIR / 'missing_values.png')
plot_pollutant_distribution(cleaned_df, 'PM2.5', FIGURES_DIR / 'pm25_distribution.png')
plot_pollutant_distribution(cleaned_df, 'NO2', FIGURES_DIR / 'no2_distribution.png')
plot_pm25_timeseries(cleaned_df, FIGURES_DIR / 'pm25_timeseries.png')
plot_correlation_matrix(corr_df, FIGURES_DIR / 'correlation_matrix.png')

print('Dataset shape')
display(shape_df)
print('Data types')
display(dtypes_df)
print('Missing values after cleaning')
display(missing_df)
print('Unique values')
display(unique_df)
print('Numeric summary')
display(numeric_df)
print('Station distribution')
display(station_df)
print('Hour distribution')
display(hour_df)
print('Time-slot distribution')
display(time_slot_df)


Dataset shape


,metric,value
0,rows,5426
1,columns,11


Data types


,column,dtype
0,DEWP,float64
1,NO2,float64
2,O3,float64
3,PM10,float64
4,PM2.5,float64
5,TEMP,float64
6,WSPM,float64
7,datetime,datetime64[us]
8,hour,int64
9,station,str


Missing values after cleaning


...

,station,rows
0,Aotizhongxin,2548
1,Changping,2878


Hour distribution


,hour,rows
0,0,220
1,1,224
2,2,217
3,3,212
4,4,205
5,5,222
6,6,224
7,7,221
8,8,217
9,9,228


Time-slot distribution


,time_slot,rows
0,night,1300
1,morning,1357
2,afternoon,1422
3,evening,1347


## Step 4-6: Candidate RFD Validation

Threshold sets are defined in `src/rfd.py` as `strict`, `medium`, and `relaxed`.
Candidate rules are evaluated on balanced 1500-row sample to keep pairwise validation tractable.


In [5]:
candidate_df, candidate_violations = run_candidate_validation(
    sample_df,
    RESULTS_DIR / 'rfd_candidate_results.csv',
)

candidate_table = candidate_df.loc[:, ['rule_label', 'support', 'confidence', 'violations', 'interpretation']].copy()
for column in ['support', 'confidence']:
    candidate_table[column] = candidate_table[column].round(4)

display(candidate_table)


,rule_label,support,confidence,violations,interpretation
2,"station, PM2.5 -> PM10",0.0734,0.4663,44020,"Within one station, particulate measures shoul..."
3,"station, TEMP, WSPM -> O3",0.0346,0.3852,23936,"Within one station, similar temperature and wi..."
4,"hour, NO2 -> PM2.5",0.0263,0.3170,20169,Similar hour and NO2 levels should often align...
0,"station, hour, TEMP -> NO2",0.0086,0.2537,7254,"Same station, similar hour and temperature sho..."
1,"station, TEMP, DEWP -> PM2.5",0.0137,0.2025,12312,"Within one station, similar thermal conditions..."


## Step 7: Lightweight Discovery

In [6]:
discovery_df = run_lightweight_discovery(
    sample_df,
    RESULTS_DIR / 'rfd_discovered_top10.csv',
)

if discovery_df.empty:
    print('No discovered RFD met support >= 0.01 and confidence >= 0.85 under medium thresholds.')
else:
    display(discovery_df.round(4))


No discovered RFD met support >= 0.01 and confidence >= 0.85 under medium thresholds.


## Step 8-9: Threshold and Station Comparisons

In [7]:
threshold_df = run_threshold_comparison(
    sample_df,
    RESULTS_DIR / 'rfd_threshold_comparison.csv',
    FIGURES_DIR / 'rfd_confidence_by_threshold.png',
)
station_df_rfd = run_station_comparison(
    sample_df,
    RESULTS_DIR / 'rfd_station_comparison.csv',
    FIGURES_DIR / 'rfd_confidence_by_station.png',
)

display(threshold_df[['rule_label', 'threshold_set', 'support', 'confidence', 'violations']].round(4))
display(station_df_rfd[['rule_label', 'station_scope', 'support', 'confidence', 'violations']].round(4))


,rule_label,threshold_set,support,confidence,violations
0,"hour, NO2 -> PM2.5",medium,0.0263,0.3170,20169
1,"hour, NO2 -> PM2.5",relaxed,0.0375,0.3829,26011
2,"hour, NO2 -> PM2.5",strict,0.0140,0.2010,12556
3,"station, PM2.5 -> PM10",medium,0.0734,0.4663,44020
4,"station, PM2.5 -> PM10",relaxed,0.1004,0.6104,43983
5,"station, PM2.5 -> PM10",strict,0.0417,0.3741,29367
6,"station, TEMP, DEWP -> PM2.5",medium,0.0137,0.2025,12312
7,"station, TEMP, DEWP -> PM2.5",relaxed,0.0271,0.2582,22596
8,"station, TEMP, DEWP -> PM2.5",strict,0.0042,0.1415,4083
9,"station, TEMP, WSPM -> O3",medium,0.0346,0.3852,23936


,rule_label,station_scope,support,confidence,violations
0,PM2.5 -> PM10,Aotizhongxin,0.1430,0.4366,22633
1,PM2.5 -> PM10,Changping,0.1507,0.4946,21387
2,"TEMP, DEWP -> PM2.5",Aotizhongxin,0.0289,0.2016,6479
3,"TEMP, DEWP -> PM2.5",Changping,0.0261,0.2036,5833
4,"TEMP, WSPM -> O3",Aotizhongxin,0.0727,0.4170,11911
5,"TEMP, WSPM -> O3",Changping,0.0659,0.3500,12025
6,"hour, NO2 -> PM2.5",Aotizhongxin,0.0250,0.3491,4570
7,"hour, NO2 -> PM2.5",Changping,0.0298,0.2927,5912
8,"hour, TEMP -> NO2",Aotizhongxin,0.0173,0.2458,3673
9,"hour, TEMP -> NO2",Changping,0.0173,0.2616,3581


## Step 10: Violation Analysis

In [8]:
selected_violations = select_top_rule_violations(candidate_df, candidate_violations, top_rules=2, per_rule=5)
violations_df = export_violation_examples(
    selected_violations,
    RESULTS_DIR / 'violations_examples.csv',
    limit=10,
)

display(violations_df)


,rule_label,lhs,rhs,row_index_1,row_index_2,datetime_1,datetime_2,station_1,station_2,rhs_value_1,rhs_value_2,rhs_abs_diff,PM2.5_1,PM2.5_2,TEMP_1,TEMP_2,WSPM_1,WSPM_2
0,"station, PM2.5 -> PM10","station, PM2.5",PM10,36,167,2013-09-01 18:00:00,2013-09-04 12:00:00,Aotizhongxin,Aotizhongxin,103.0,44.0,59.0,43.0,34.0,NaN,NaN,NaN,NaN
1,"station, PM2.5 -> PM10","station, PM2.5",PM10,36,236,2013-09-01 18:00:00,2013-09-05 23:00:00,Aotizhongxin,Aotizhongxin,103.0,44.0,59.0,43.0,39.0,NaN,NaN,NaN,NaN
2,"station, PM2.5 -> PM10","station, PM2.5",PM10,36,240,2013-09-01 18:00:00,2013-09-06 01:00:00,Aotizhongxin,Aotizhongxin,103.0,48.0,55.0,43.0,39.0,NaN,NaN,NaN,NaN
3,"station, PM2.5 -> PM10","station, PM2.5",PM10,36,245,2013-09-01 18:00:00,2013-09-06 04:00:00,Aotizhongxin,Aotizhongxin,103.0,53.0,50.0,43.0,43.0,NaN,NaN,NaN,NaN
4,"station, PM2.5 -> PM10","station, PM2.5",PM10,36,247,2013-09-01 18:00:00,2013-09-06 05:00:00,Aotizhongxin,Aotizhongxin,103.0,46.0,57.0,43.0,45.0,NaN,NaN,NaN,NaN
5,"station, TEMP, WSPM -> O3","station, TEMP, WSPM",O3,36,129,2013-09-01 18:00:00,2013-09-03 17:00:00,Aotizhongxin,Aotizhongxin,176.0,195.0,19.0,NaN,NaN,26.1,26.1,1.0,1.8
6,"station, TEMP, WSPM -> O3","station, TEMP, WSPM",O3,36,133,2013-09-01 18:00:00,2013-09-03 19:00:00,Aotizhongxin,Aotizhongxin,176.0,146.0,30.0,NaN,NaN,26.1,24.6,1.0,0.8
7,"station, TEMP, WSPM -> O3","station, TEMP, WSPM",O3,36,137,2013-09-01 18:00:00,2013-09-03 21:00:00,Aotizhongxin,Aotizhongxin,176.0,116.0,60.0,NaN,NaN,26.1,24.3,1.0,0.9
8,"station, TEMP, WSPM -> O3","station, TEMP, WSPM",O3,36,139,2013-09-01 18:00:00,2013-09-03 22:00:00,Aotizhongxin,Aotizhongxin,176.0,95.0,81.0,NaN,NaN,26.1,24.1,1.0,1.0
9,"station, TEMP, WSPM -> O3","station, TEMP, WSPM",O3,36,218,2013-09-01 18:00:00,2013-09-05 14:00:00,Aotizhongxin,Aotizhongxin,176.0,92.0,84.0,NaN,NaN,26.1,27.4,1.0,0.7


## Takeaways

- Candidate RFDs are interpretable but moderately weak on this subset; none behaves like near-deterministic rule.
- Relaxed thresholds increase confidence and support, but still do not yield discovery results above project filter (`0.01`, `0.85`).
- Violating pairs remain useful for UDT framing: they can indicate local inconsistency, unusual pollution dynamics, or missing contextual variables.
- This is approximate consistency analysis, not prediction and not causality.
